# Retrieval of ClueWeb09 Category B documents based on BM25 via OpenSearch

Dedicated notebook for ClueWeb09: the index is title+text like the other
titled corpora, but two treatments are ClueWeb-specific --

1. **Diversity topics** (TREC Web Track 2009-2012): topics carry `.query`
   text and subtopics, judged per subtopic.
2. **Post-retrieval near-duplicate removal** using the Webis **CopyCat**
   duplicate groups (see below) -- applied after retrieval so dedup stays a
   switchable experimental condition rather than a baked-in index choice.

#### Configuration

In [ ]:
index_name = "clueweb09_catb_bm25"
q = "obama family tree"
dataset_name = "clueweb09/catb/trec-web-2009/diversity"

# CopyCat near-duplicate groups (Webis): precomputed for ClueWeb09 -- no
# detection run needed. Download from the Webis data page / chatnoir-copycat
# repository and set the path; expected format: one group per line, docids
# separated by whitespace. Leave the path missing to skip dedup (no-op).
copycat_groups_path = "~/.ir_datasets/clueweb09/copycat-groups.txt"

In [ ]:
import sys
!{sys.executable} -m pip install -q ir_datasets opensearch-py dotenv

In [ ]:
import pprint

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [ ]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

#### BM25 Search

In [ ]:
def build_query(query: str) -> dict:
    return {
        "multi_match": {
            "query": query,
            "fields": ["title^2", "text"]  # title gets a boost
        }
    }

def search(query: str, size: int = 10) -> dict:
    return client.search(index=index_name, body={
        "size": size,
        "_source": ["docid", "title", "text"],
        "query": build_query(query),
    })

def show(resp_or_hits, label=""):
    hits = resp_or_hits["hits"]["hits"] if isinstance(resp_or_hits, dict) else resp_or_hits
    print(f"\nTop {len(hits)} hits{' (' + label + ')' if label else ''}\n")
    for hit in hits:
        src = hit["_source"]
        print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")

In [ ]:
show(search(q, size=5), q)

#### Search with a Topic from the Dataset

Diversity topics carry the search text in `.query` (not `.title`) plus the
judged subtopics.

In [ ]:
import ir_datasets
dataset = ir_datasets.load(dataset_name)

In [ ]:
topic = next(dataset.queries_iter())
pprint.pprint(topic)
topic_query = topic.query
show(search(topic_query, size=5), f"topic {topic.query_id}: {topic_query}")

#### Post-retrieval near-duplicate removal (CopyCat)

ClueWeb09 contains many near-duplicate pages; they inflate redundancy in
SERPs and are degenerate "diversity" failures that pollute alpha-nDCG /
ERR-IA measurement (Frobe et al., SIGIR 2020; CopyCat, CIKM 2021). The
Webis CopyCat resource ships **precomputed** near-duplicate groups for
ClueWeb09, so no detection run is needed.

Dedup here is a post-retrieval filter: over-fetch, then keep only the
highest-ranked document of each duplicate group. Retrieval itself is
untouched, so deduplicated and raw runs stay comparable side by side.

In [ ]:
# docid -> group id, loaded once. Docs absent from every group are their own
# singleton group (kept as-is).
dup_group = {}
_path = os.path.expanduser(copycat_groups_path)
if os.path.isfile(_path):
    with open(_path) as f:
        for gid, line in enumerate(f):
            for docid in line.split():
                dup_group[docid] = gid
    print(f"loaded {len(dup_group)} docids in {gid + 1} duplicate groups")
else:
    print(f"{_path} not found -- dedup will be a no-op")

In [ ]:
def dedup(hits: list) -> list:
    """Keep the highest-ranked hit of each duplicate group (rank order kept)."""
    seen, out = set(), []
    for hit in hits:
        group = dup_group.get(hit["_id"], hit["_id"])   # singleton fallback
        if group in seen:
            continue
        seen.add(group)
        out.append(hit)
    return out

def search_dedup(query: str, size: int = 10, overfetch: int = 5) -> list:
    """Over-fetch size*overfetch, dedup, return the top `size` survivors."""
    resp = search(query, size=size * overfetch)
    return dedup(resp["hits"]["hits"])[:size]

show(search(topic_query, size=5), "raw")
show(search_dedup(topic_query, size=5), "deduplicated")

#### Rerank with the cross-encoder (common second stage)

Re-run the same first-stage query through the `rerank_bge_m3` search pipeline
(`BAAI/bge-reranker-v2-m3`, multilingual -- see
[ml_model_registration.ipynb](../indexing/opensearch/ml_model_registration.ipynb)).
Works identically over every index and ranker; combine with `dedup()` by
applying it to the reranked hits.

**Gotcha:** the request must return `_source` including the `text` field --
the rerank processor reads `document_fields: ["text"]` from `_source`; without
it every hit gets the same score and the order silently stays unchanged.

In [ ]:
def search_reranked(query: str, size: int = 10, rerank_pipeline: str = "rerank_bge_m3") -> dict:
    """Same first stage as search(), then cross-encoder reranking server-side."""
    return client.search(
        index=index_name,
        params={"search_pipeline": rerank_pipeline},
        body={
            "size": size,
            "_source": ["docid", "title", "text"],   # MUST include "text" (rerank context)
            "query": build_query(query),
            "ext": {"rerank": {"query_context": {"query_text": query}}},
        },
    )

show(search(q, size=5), "first stage")
show(search_reranked(q, size=5), "reranked")
show(dedup(search_reranked(q, size=25)["hits"]["hits"])[:5], "reranked + deduplicated")